In [2]:
import os, shutil, math, glob
from math import sin,cos,sqrt,acos,asin
from locale import atof, atoi

parent_dir = '/storage/home/hcoda1/5/sjamdade3/scratch/August_13_2023/water_screening/MD_MC/Manuscript_Final/Defects/'

path_general_files = '/storage/home/hcoda1/5/sjamdade3/scratch/August_13_2023/water_screening/MD_MC/Manuscript_Final/Defects/SOP_files/'

MOF_type = 'defects/'

In [3]:
##### Create box.xyz and HISTORY.XYZ

path_CIFs = os.path.join(parent_dir, 'MOFs', str(MOF_type))

path_MD_data = os.path.join(path_CIFs, 'MD_data')

path_python_files = os.path.join(path_general_files, 'python_files/')

path_MOF_NVT_MC = os.path.join(path_CIFs, 'NVT_MC/')


for file in os.listdir(path_MD_data):
    d = os.path.join(path_MD_data, file)
    if os.path.isdir(d):
        mof = str(file)
        
            
        ## Create  box.xyz
        
        with open(os.path.join(path_MOF_NVT_MC, str(mof), 'Movies', 'System_0', 'Framework_0_initial.vasp'),'r') as f_cell, open(os.path.join(path_MD_data, str(mof), 'box.xyz'), 'a') as secondfile:
            l = 0
            lines=f_cell.readlines()
            f_cell.close()
            for line in lines:
                row=line.split()

                if not line.strip():
                    continue

                l = l + 1    
                if l == 3:
                    secondfile.write(line)
                if l == 4:
                    secondfile.write(line)
                if l == 5:
                    secondfile.write(line)

        
        ## Create  HISTORY.xyz
        
        with open(os.path.join(path_MD_data, str(mof), 'nvt_framework.xyz'),'r') as firstfile, open(os.path.join(path_MD_data, str(mof), 'HISTORY.xyz'), 'a') as f_history:
                lines=firstfile.readlines()
                firstfile.close()
                L = 0 
                a = 1000000000000000000000000000000
                for line in lines:
                    L = L + 1
                    row=line.split()

                    if not line.strip():
                        continue
                    
                    if L == 1 and len(row) == 1:
                        f_history.write(line)
                        
                    if len(row) == 3 and row[0] == 'Atoms.' and row[1] == 'Timestep:' and row[2] == '1000000':
                        a = L
                        f_history.write(line)

                    if L > a and len(row) == 4:                                                                                                                                                                                                                                       
                        f_history.write(line)
                        
        ## Create  WATER.xyz
        
        with open(os.path.join(path_MD_data, str(mof), 'water_coordinates.xyz'),'r') as firstfile, open(os.path.join(path_MD_data, str(mof), 'WATER.xyz'), 'a') as f_water:
        
                lines=firstfile.readlines()
                firstfile.close()
                L = 0 
                a = 100000000000000
                for line in lines:
                    L = L + 1
                    row=line.split()

                    if not line.strip():
                        continue

                    if L == 1 and len(row) == 1:
                        f_water.write(line)

                    if len(row) == 3 and row[0] == 'Atoms.' and row[1] == 'Timestep:' and row[2] == '1000000':
                        a = L
                        f_water.write(line)

                    if L > a and len(row) == 4:                                                                                                                                                                                                                                       
                        f_water.write(line)
        
        ###Creating 111 MOF CIF file 
        
        path_xyz = os.path.join(path_MD_data, str(mof))
        
        #basic functions
        def theta(vec1,vec2):
            theta = acos((vec1[0]*vec2[0] + vec1[1]*vec2[1] + vec1[2]*vec2[2])/((modulus(vec1))*modulus(vec2)))
            return math.degrees(theta)

        #get the mod of vector
        def modulus(vec1):
            return float(sqrt(vec1[0]*vec1[0] + vec1[1]*vec1[1] + vec1[2]*vec1[2]))

        #open necessary files
        box=open(str(path_xyz)+'/box.xyz','r')
        xyz=open(str(path_xyz)+'/HISTORY.xyz','r')

        frame = 0
        while 1:
            # control
            line = box.readline()
            if not line: break

            #lattice vectors
            a1,b1,c1 = line.split()
            a2,b2,c2 = box.readline().split()
            a3,b3,c3 = box.readline().split()
            lata = [ atof(a1), atof(b1), atof(c1)]
            latb = [ atof(a2), atof(b2), atof(c2)]
            latc = [ atof(a3), atof(b3), atof(c3)]

            #lattice parameters
            a = modulus(lata)
            b = modulus(latb)
            c = modulus(latc)
            alpha = theta(latb,latc)
            beta = theta(lata,latc)
            gam = theta(lata,latb)

            print (" a b c alpha beta gamma",a,b,c,alpha,beta,gam)
            # print " frame ", frame

            #radians needed for sin, cos
            alpha = math.radians(alpha)
            beta = math.radians(beta)
            gam = math.radians(gam)

            frac = [0,0,0]
            cart = [0,0,0]

            #reqd constants for cart2frac matrix
            sa = sin(alpha)
            sb = sin(beta)
            sg = sin(gam)
            ca = cos(alpha)
            cb = cos(beta)
            cg = cos(gam)

            # v = a*b*c*sqrt(1-ca*ca-cb*cb-cg*cg+ 2*ca*cb*cg)
            v = sqrt(1-ca*ca-cb*cb-cg*cg+ 2*ca*cb*cg)

            # conversion matrix
            ## cart2frac = [ [1/a,0,0], [-cg/(a*sg), (1/(b*sg)), 0], [(ca*cg-cb)/(a*v*sg), ((cb*cg-ca)/(b*v*sg)), sg/(c*v)] ]
            cart2frac = [ [1/a,-cg/(a*sg),(ca*cg-cb)/(a*v*sg)], [0,(1/(b*sg)),((cb*cg-ca)/(b*v*sg))], [0,0,sg/(c*v)] ]

            #get number of atoms
            nat = xyz.readline()
            nat = atoi(nat)
            xyz.readline() # skip comment
            #get cartesian coordinates ... C 1.234 1.234 1.234
            X_f = []
            Y_f = []
            Z_f = []
            for index in range(nat):
                cart = [ 0,0,0 ]
                lab,x,y,z = xyz.readline().split()
                cart = [atof(x),atof(y),atof(z)]

                #conversion
                frac = [0.0,0.0,0.0]
                for i in range(0,3):
                    for j in range(0,3):
                        frac[i] = cart2frac[i][j]*cart[j] + frac [i]

                X_f.append(frac[0]) 
                Y_f.append(frac[1]) 
                Z_f.append(frac[2])        

                ## print lab, '%.2f %.2f %.2f' % (frac[0], frac[1], frac[2]) 
                ## print '%.2f %.2f %.2f' % (frac[0], frac[1], frac[2]) 
                #print ('%14.5f %14.5f %14.5f' % (frac[0], frac[1], frac[2]))
            frame = frame + 1

        box.close()
        xyz.close()

        for f_CIF in os.listdir(os.path.join(path_MD_data, str(mof))):

            if f_CIF.startswith('data.') and f_CIF.endswith('.cif'):
                f_CIF_Array = f_CIF.split("data.")
                f_CIF_file = f_CIF_Array[1] 

        n = -1
        
        for f_m in os.listdir(os.path.join(path_MOF_NVT_MC, str(mof), 'Movies','System_0')):
            
            if f_m.startswith('Framework_0_final') and f_m.endswith('_P1.cif'):
                
                f_CIF_template = str(f_m)
                
        with open(os.path.join(path_MOF_NVT_MC, str(mof), 'Movies','System_0', str(f_CIF_template)),'r') as firstfile, open(str(path_xyz)+'/'+str(f_CIF_file),'a') as secondfile:

            for line in firstfile:

                row=line.split()

                if not line.strip():
                    continue 

                if not len(row) == 6:
                    secondfile.write(line)

                if len(row) == 6 and row[0] == '_chemical_name_common' :
                    secondfile.write(line)

                if len(row) == 6 and not row[0] == '_chemical_name_common' :
                    n = n + 1
                    _atom_site_label = str(row[0])
                    _atom_site_type_symbol =str(row[1])
                    _atom_site_charge = str(row[5])

                    xyz_line = str(_atom_site_label) + '   ' + str(_atom_site_type_symbol) + '   ' + str(X_f[n]) + '   ' + str(Y_f[n]) + '   ' + str(Z_f[n])  + '   ' + str(_atom_site_charge) + os.linesep 
                    #print(xyz_line)
                    secondfile.write(str(xyz_line))

                    
        ### Create restart file from WATER.XYZ

        x_O = []
        y_O = []
        z_O = []

        x_H1 = []
        y_H1 = []
        z_H1 = []

        x_H2 = []
        y_H2 = []
        z_H2 = []

        x_L = []
        y_L = []
        z_L = []

        f_water = open(str(path_xyz)+'/'+'WATER.xyz', "rt")
        lines = f_water.readlines()
        l = 1
        for line in lines:

            row=line.split()

            if not line.strip():
                continue

            if l == 2:
                m = 1

            if row[0] == 'O' and len(row) == 4:
                l = 0
                m = 0
                x_O.append(float(row[1])) 
                y_O.append(float(row[2])) 
                z_O.append(float(row[3])) 
                l = l + 1

            if row[0] == 'H' and len(row) == 4 and l == 1 :
                x_H1.append(float(row[1])) 
                y_H1.append(float(row[2])) 
                z_H1.append(float(row[3]))  
                l = l + 1

            if row[0] == 'H' and len(row) == 4 and m == 1:
                x_H2.append(float(row[1])) 
                y_H2.append(float(row[2])) 
                z_H2.append(float(row[3]))     

                A1 = -x_O[-1] + x_H1[-1]
                B1 = -y_O[-1] + y_H1[-1]
                C1 = -z_O[-1] + z_H1[-1]
                
                d1 = math.sqrt(A1**2 + B1**2 + C1**2)
                x_H1[-1] = x_O[-1] + 0.9572*A1/d1
                y_H1[-1] = y_O[-1] + 0.9572*B1/d1
                z_H1[-1] = z_O[-1] + 0.9572*C1/d1                
                

                A2 = -x_O[-1] + x_H2[-1]
                B2 = -y_O[-1] + y_H2[-1]
                C2 = -z_O[-1] + z_H2[-1]
                
                d2 = math.sqrt(A2**2 + B2**2 + C2**2)
                x_H2[-1] = x_O[-1] + 0.9572*A2/d2
                y_H2[-1] = y_O[-1] + 0.9572*B2/d2
                z_H2[-1] = z_O[-1] + 0.9572*C2/d2                

                A = (A1 + A2)
                B = (B1 + B2)
                C = (C1 + C2)
                d = math.sqrt(A**2 + B**2 + C**2)

                a = 0.15*A/d + x_O[-1]
                b = 0.15*B/d + y_O[-1]
                c = 0.15*C/d + z_O[-1]

                x_L.append(a) 
                y_L.append(b) 
                z_L.append(c)     
        
        n = 0
        
        for f_m in os.listdir(os.path.join(path_MOF_NVT_MC, str(mof), 'Restart','System_0')):
            
                f_restart_template = str(f_m)
                
        with open(os.path.join(path_MOF_NVT_MC, str(mof), 'Restart','System_0', str(f_restart_template)),'r') as firstfile, open(str(path_xyz) + '/' + str(f_restart_template),'a') as secondfile:

            for line in firstfile:

                row=line.split()

                if not line.strip():
                    secondfile.write(line)
                    continue 

                if not row[0] == 'Adsorbate-atom-position:':
                    secondfile.write(line)

                if len(row) == 6 and row[0] == 'Adsorbate-atom-position:':

                    if int(row[2]) == 0:
                        x = x_O
                        y = y_O
                        z = z_O

                    if int(row[2]) == 1:
                        x = x_H1
                        y = y_H1
                        z = z_H1        

                    if int(row[2]) == 2:
                        x = x_H2
                        y = y_H2
                        z = z_H2    

                    if int(row[2]) == 3:
                        x = x_L
                        y = y_L
                        z = z_L   

                    xyz_line = str(row[0]) + '   ' + str(row[1]) + '   ' + str(row[2]) +'   ' + str(x[int(row[1])]) + '   ' + str(y[int(row[1])]) + '   ' + str(z[int(row[1])]) + os.linesep 
                    secondfile.write(str(xyz_line))
            

        fin = open(str(path_xyz)+ '/' + str(f_restart_template), "rt")

        lines=fin.readlines()

        for line in lines:
            row=line.split()

            if not line.strip():
                continue
                
            if row[0] == 'number-of-unit-cells:' and len(row) == 4:
                
                nA = float(row[1])
                nB = float(row[2])
                nC = float(row[3])                
                
            if row[0] == 'unit-cell-vector-a:':
                a1 = float(row[1])
                a2 = float(row[2])
                a3 = float(row[3])
            if row[0] == 'unit-cell-vector-b:':
                b1 = float(row[1])
                b2 = float(row[2])
                b3 = float(row[3])
            if row[0] == 'unit-cell-vector-c:':
                c1 = float(row[1])
                c2 = float(row[2])
                c3 = float(row[3])

            if row[0] == 'cell-vector-a:':
                A1 = float(row[1])
                A2 = float(row[2])
                A3 = float(row[3])
                
                a1 = A1
                a2 = A2
                a3 = A3

            if row[0] == 'cell-vector-b:':
                B1 = float(row[1])
                B2 = float(row[2])
                B3 = float(row[3])
                
                b1 = B1
                b2 = B2
                b3 = B3

            if row[0] == 'cell-vector-c:':
                C1 = float(row[1])
                C2 = float(row[2])
                C3 = float(row[3])
                
                c1 = C1
                c2 = C2
                c3 = C3
                
            if row[0] == 'cell-lengths:': 
                
                LA = nA*float(row[1])
                LB = nB*float(row[2])
                LC = nC*float(row[3])
                
        fin.close() 
        
       ## write 5 lines         

        with open(str(path_xyz)+ '/' + str(f_restart_template), "rt") as firstfile, open(str(path_xyz) + '/' + str(f_restart_template)+'_new','a') as secondfile:
        
            for line in firstfile:

                row=line.split()

                if not line.strip():
                    secondfile.write(line)
                    continue 

                if row[0] == 'number-of-unit-cells:':
                    restart_line = 'number-of-unit-cells: 1 1 1'
                    secondfile.write(str(restart_line))
                    continue 
                    

                if len(row) == 4 and row[0] == 'unit-cell-vector-a:':
                    restart_line = os.linesep +'unit-cell-vector-a:'+'     '+str(a1)+'     '+str(a2)+'     '+str(a3)
                    secondfile.write(str(restart_line))
                    continue
                    
                if len(row) == 4 and row[0] == 'unit-cell-vector-b:':    
                    restart_line = os.linesep +'unit-cell-vector-b:'+'     '+str(b1)+'     '+str(b2)+'     '+str(b3)
                    secondfile.write(str(restart_line))
                    continue
                    
                if len(row) == 4 and row[0] == 'unit-cell-vector-c:':    
                    restart_line = os.linesep +'unit-cell-vector-c:'+'     '+str(c1)+'     '+str(c2)+'     '+str(c3) + os.linesep
                    secondfile.write(str(restart_line))
                    continue
                
                if len(row) == 4 and row[0] == 'cell-lengths:':
                    restart_line = 'cell-lengths:'+'     '+str(LA)+'     '+str(LB)+'     '+str(LC) + os.linesep
                    secondfile.write(str(restart_line))
                    continue
                    
                else: 
                    secondfile.write(line)

        os.remove(str(path_xyz) + '/' + str(f_restart_template))            
        os.rename(str(path_xyz) + '/' + str(f_restart_template)+'_new', str(path_xyz) + '/' + str(f_restart_template))
        

        



 a b c alpha beta gamma 29.678132 29.567218 30.53622999997904 118.98032400002177 119.26481600002204 90.50792700000001
 a b c alpha beta gamma 24.4719 30.300786 32.86178199999636 99.24688700000104 91.7075580000002 109.302933
 a b c alpha beta gamma 24.802647 28.642167 36.46814700003675 88.64796400000135 78.91863300001131 79.84623
 a b c alpha beta gamma 26.026602 26.0266 27.844000000005025 106.37300099999696 106.37300099999696 91.273399
 a b c alpha beta gamma 25.618988 21.259383 49.73216399995899 106.55943300001404 98.64370000000717 89.970581
 a b c alpha beta gamma 23.307257999999997 22.492194 40.772415000020594 114.45416999998683 103.47775999999308 90.029167
 a b c alpha beta gamma 25.618988 28.345844 49.73216200003925 106.55943299998656 98.64369999999313 89.970581
 a b c alpha beta gamma 29.5113 24.7902 38.5695 90.0 90.0 101.217
 a b c alpha beta gamma 23.307257999999997 44.984386 40.77241500000396 114.45416999999748 103.47775999999868 90.029175
 a b c alpha beta gamma 26.026602 26.